In [1]:
# ==============================================================================
# Final Meta-Model for Production-Ready Intrusion Detection
#
# Project: Final Model Fusion (Stacking Ensemble)
# Author: B12 - BPPIMT - CSE 21-25
# Date: 12-06-25
#
# Description:
#   This script builds a "Level 1" meta-model (Logistic Regression) that learns
#   from the predictions of two "Level 0" base models: a multi-class ANN and a
#   high-recall binary LSTM+Attention model. This stacking approach creates a
#   final classifier that is more robust and accurate than either base model alone.
# ==============================================================================

# ==============================================================================
# 1. SETUP: LIBRARIES AND ENVIRONMENT
# ==============================================================================
import os
import gc
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
print(f"TensorFlow Version: {tf.__version__}")


# ==============================================================================
# 2. LOAD ALL MODELS AND ARTIFACTS
# ==============================================================================
# --- Define Paths ---
ANN_PATH = '/kaggle/input/cse-cic-ids2018-nids-ann/ann_multiclass_model/'
LSTM_PATH = '/kaggle/input/time-series-lstm-multi-head-attention/final_tuned_model/'
SOURCE_DATA_PATH = '/kaggle/input/test-set-gen-cic-ids-2018/meta_model_data/'
EXPORT_PATH = './meta_model_artifacts/'
os.makedirs(EXPORT_PATH, exist_ok=True)
RANDOM_STATE = 42

# --- Load ANN Model ("Snapshot Specialist") ---
print("\n[INFO] Loading Multi-Class ANN Model and artifacts...")
ann_model = tf.keras.models.load_model(os.path.join(ANN_PATH, 'best_threat_detector_model.h5'))
ann_scaler = joblib.load(os.path.join(ANN_PATH, 'scaler.pkl'))
with open(os.path.join(ANN_PATH, 'feature_names.json'), 'r') as f:
    ANN_ORIGINAL_FEATURES = json.load(f)

# --- Load LSTM Model ("Context Specialist") ---
print("[INFO] Loading Binary LSTM+Attention Model and artifacts...")
lstm_model = tf.keras.models.load_model(os.path.join(LSTM_PATH, 'intrusion_detection_model.keras'))
lstm_scaler = joblib.load(os.path.join(LSTM_PATH, 'scaler.pkl'))
with open(os.path.join(LSTM_PATH, 'feature_mapping.json'), 'r') as f:
    LSTM_TOP_FEATURES = json.load(f)
with open(os.path.join(LSTM_PATH, 'model_config.json'), 'r') as f:
    lstm_config = json.load(f)
WINDOW_LENGTH = lstm_config['window_length']

# --- Load the Source Data (The held-out test set for meta-feature generation) ---
print("[INFO] Loading the source data...")
X_source = pd.read_parquet(os.path.join(SOURCE_DATA_PATH, 'X_test_raw.parquet'))
y_source = pd.read_csv(os.path.join(SOURCE_DATA_PATH, 'y_test_raw.csv')).squeeze("columns")

print("All models and data loaded successfully.")


# ==============================================================================
# 3. LEVEL 0 PREDICTION: GENERATE META-FEATURES (MEMORY-SAFE)
# ==============================================================================
print("\n[INFO] Step 3: Generating predictions from base models to create meta-features...")

# --- Get Predictions from ANN Model (This is fast and memory-safe) ---
print("  - Getting predictions from ANN model...")
X_source_for_ann = X_source.reindex(columns=ANN_ORIGINAL_FEATURES, fill_value=0)
X_scaled_for_ann = ann_scaler.transform(X_source_for_ann)
ann_preds = ann_model.predict(X_scaled_for_ann, batch_size=4096)
print(f"    ANN predictions generated. Shape: {ann_preds.shape}")

# --- Get Predictions from LSTM Model (Using a memory-safe generator) ---
print("  - Getting predictions from LSTM model using a batch generator...")
X_filtered_for_lstm = X_source[LSTM_TOP_FEATURES]
X_scaled_for_lstm = lstm_scaler.transform(X_filtered_for_lstm)

def prediction_generator(data, window_length, batch_size):
    """
    A generator to yield windowed batches of data for prediction,
    preventing memory overload.
    """
    num_samples = data.shape[0]
    num_batches = int(np.ceil((num_samples - window_length + 1) / batch_size))

    for i in range(num_batches):
        start_index = i * batch_size
        end_index = start_index + batch_size
        
        # This batch will have overlapping windows
        # We need a slightly larger slice of the original data to create these windows
        data_slice = data[start_index : end_index + window_length -1]
        
        if len(data_slice) < window_length:
            break

        X_batch_windows = []
        for j in range(len(data_slice) - window_length + 1):
            X_batch_windows.append(data_slice[j:j+window_length])
        
        if not X_batch_windows:
            break
            
        yield np.array(X_batch_windows)

# A stride of 1 is implicitly used here to maximize data for the meta-model
# This means our aligned indices will just be from WINDOW_LENGTH-1 to the end
num_lstm_preds = len(X_scaled_for_lstm) - WINDOW_LENGTH + 1
aligned_indices = np.arange(WINDOW_LENGTH - 1, len(X_scaled_for_lstm))

# Create the generator
lstm_pred_gen = prediction_generator(X_scaled_for_lstm, WINDOW_LENGTH, batch_size=4096)

# Use the generator to predict in batches
print("    Predicting with LSTM in batches...")
lstm_preds_list = []
for batch in lstm_pred_gen:
    lstm_preds_list.append(lstm_model.predict(batch, verbose=0))

lstm_preds = np.vstack(lstm_preds_list)
# We might have a few extra predictions due to batching, so we trim to the exact size
lstm_preds = lstm_preds[:num_lstm_preds]

print(f"    LSTM predictions generated. Shape: {lstm_preds.shape}")

# --- Align all datasets ---
print("  - Aligning datasets for meta-model input...")
ann_preds_aligned = ann_preds[aligned_indices]
y_true_aligned = y_source.iloc[aligned_indices]

print("Alignment complete.")
del X_source, y_source, X_source_for_ann, X_scaled_for_ann, X_filtered_for_lstm, X_scaled_for_lstm, ann_preds; gc.collect()



# ==============================================================================
# 4. CREATE AND EXPORT META-DATASET
# ==============================================================================
print("\n[INFO] Step 4: Creating and exporting the meta-dataset...")
X_meta = np.concatenate([ann_preds_aligned, lstm_preds], axis=1)
y_meta_true_binary = np.where(y_true_aligned == 'Benign', 0, 1)

print(f"Meta-dataset created. Features shape: {X_meta.shape}")

# --- Export the predictions and labels for auditability and reuse ---
np.save(os.path.join(EXPORT_PATH, 'ann_predictions.npy'), ann_preds_aligned)
np.save(os.path.join(EXPORT_PATH, 'lstm_predictions.npy'), lstm_preds)
np.save(os.path.join(EXPORT_PATH, 'aligned_true_labels.npy'), y_meta_true_binary)
print("✅ Saved base model predictions and true labels.")
del ann_preds_aligned, lstm_preds; gc.collect()


# ==============================================================================
# 5. TRAIN AND EVALUATE THE META-MODEL
# ==============================================================================
print("\n[INFO] Step 5: Training and evaluating the final meta-model...")

# --- Split the meta-dataset for honest evaluation ---
X_meta_train, X_meta_test, y_meta_train, y_meta_test = train_test_split(
    X_meta, y_meta_true_binary, test_size=0.2, random_state=RANDOM_STATE, stratify=y_meta_true_binary
)

# --- Train the Meta-Model ("The Manager") ---
print("Training the final Logistic Regression meta-model...")
meta_model = LogisticRegression(random_state=RANDOM_STATE, solver='liblinear', class_weight='balanced', max_iter=1000)
meta_model.fit(X_meta_train, y_meta_train)
print("Meta-model training complete.")

# --- Save the final meta-model ---
joblib.dump(meta_model, os.path.join(EXPORT_PATH, 'meta_model.pkl'))
print(f"✅ Final meta-model saved to '{EXPORT_PATH}'")

# --- Evaluate the Meta-Model ---
y_pred_meta = meta_model.predict(X_meta_test)
final_report = classification_report(y_meta_test, y_pred_meta, target_names=['BENIGN', 'ATTACK'], output_dict=True)
print("\n--- FINAL META-MODEL CLASSIFICATION REPORT ---")
print(pd.DataFrame(final_report).transpose())
with open(os.path.join(EXPORT_PATH, 'meta_model_classification_report.json'), 'w') as f:
    json.dump(final_report, f, indent=4)

final_cm = confusion_matrix(y_meta_test, y_pred_meta)
plt.figure(figsize=(8, 6))
sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['BENIGN', 'ATTACK'], yticklabels=['BENIGN', 'ATTACK'])
plt.title('Final Meta-Model Confusion Matrix', fontsize=16)
plt.ylabel('Actual Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.savefig(os.path.join(EXPORT_PATH, 'meta_model_confusion_matrix.png')); plt.close()
print("✅ Final confusion matrix saved.")

print("\n[SUCCESS] Meta-model training and evaluation complete.")

2025-06-12 12:28:05.649207: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749731285.854460      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749731285.916887      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow Version: 2.18.0

[INFO] Loading Multi-Class ANN Model and artifacts...


I0000 00:00:1749731298.052258      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1749731298.052898      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[INFO] Loading Binary LSTM+Attention Model and artifacts...
[INFO] Loading the source data...
All models and data loaded successfully.

[INFO] Step 3: Generating predictions from base models to create meta-features...
  - Getting predictions from ANN model...


I0000 00:00:1749731306.536613      62 service.cc:148] XLA service 0x7bad342752e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1749731306.537403      62 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1749731306.537422      62 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1749731306.616949      62 cuda_dnn.cc:529] Loaded cuDNN version 90300


 97/470 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

I0000 00:00:1749731307.025974      62 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


470/470 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
    ANN predictions generated. Shape: (1925030, 14)
  - Getting predictions from LSTM model using a batch generator...
    Predicting with LSTM in batches...
    LSTM predictions generated. Shape: (1924991, 1)
  - Aligning datasets for meta-model input...
Alignment complete.

[INFO] Step 4: Creating and exporting the meta-dataset...
Meta-dataset created. Features shape: (1924991, 15)
✅ Saved base model predictions and true labels.

[INFO] Step 5: Training and evaluating the final meta-model...
Training the final Logistic Regression meta-model...
Meta-model training complete.
✅ Final meta-model saved to './meta_model_artifacts/'

--- FINAL META-MODEL CLASSIFICATION REPORT ---
              precision    recall  f1-score        support
BENIGN         0.974547  0.981852  0.978186  275072.000000
ATTACK         0.953720  0.935830  0.944690  109927.000000
accuracy       0.968712  0.968712  0.968712       0.968712
macro avg      0.964133  0.958841  0.96